In [420]:
import pandas as pd
import ast

PATH = "../data/raw/listings_raw.csv"
REPORTS = "../reports/"
OUTPUT = "../data/raw/listings.csv"


In [421]:
df = pd.read_csv(PATH)

In [422]:
df.isna().sum()

url               0
description       0
fetch_date        0
title             0
location          0
house_type       18
bathrooms         1
bedrooms          1
properties        0
amenities      2233
price             0
dtype: int64

In [423]:
def get_loc(loc: pd.DataFrame):
    parts = str(loc).split(",")
    if len(parts) >= 3:
        return parts[1].strip()
    return parts[0]


In [424]:
df["location"] = df["location"].apply(get_loc)

In [425]:
def fix_ht(df):
    mht = df["house_type"].isna().sum()
    print(f"Missing house_type data before update: {mht}")
    df.loc[df["house_type"].isna(), "house_type"] = "Bedsitter"
    mht = df["house_type"].isna().sum()
    print(f"Missing house_type data after update nans: {mht}")
    return df

In [426]:
df = fix_ht(df)

Missing house_type data before update: 18
Missing house_type data after update nans: 0


In [427]:
def fix_bednbath(df):
    print(f"Missing bathroom data before update: {df['bathrooms'].isna().sum()}")
    print(f"Missing bedroom data before update: {df['bedrooms'].isna().sum()}")

    df.dropna(subset=["bathrooms"], inplace=True)
    df.dropna(subset=["bedrooms"], inplace=True)

    df["bathrooms"] = df["bathrooms"].str.split(" ").str.get(0).str.strip()
    df["bedrooms"] = df["bedrooms"].str.split(" ").str.get(0).str.strip()

    print(f"Missing bathroom data after update: {df['bathrooms'].isna().sum()}")
    print(f"Missing bedroom data after update: {df['bedrooms'].isna().sum()}")

    return df

In [428]:
df = fix_bednbath(df)

Missing bathroom data before update: 1
Missing bedroom data before update: 1
Missing bathroom data after update: 0
Missing bedroom data after update: 0


In [429]:
def extract_all_attributes(df, drop=True):
    """Expand the property column into separate columns"""
    olen = len(df.columns)

    df["properties"] = df["properties"].apply(ast.literal_eval)
    properties = df["properties"].apply(pd.Series)
    df_prop = df.join(properties)
    if drop:
        df_prop = df_prop.drop("properties", axis=1)

    plen = len(df_prop.columns)

    print(f"Properties extracted; added {plen - olen} new attributes!")

    amenities = df_prop["amenities"].str.strip().str.get_dummies(sep=",")
    df_amen = df_prop.join(amenities)
    if drop:
        df_amen = df_amen.drop("amenities", axis=1)

    alen = len(df_amen.columns)
    print(f"Ameneties extracted; added {alen - plen} new attributes!")
    print(f"{alen} final attributes after extraction")

    facilities = df_amen["Facilities"].str.strip().str.get_dummies(sep=",")
    df_amen.update(facilities)  # Updates overlapping columns in-place
    df_fac = df_amen

    return df_fac

In [430]:
df = extract_all_attributes(df)

Properties extracted; added 23 new attributes!
Ameneties extracted; added 17 new attributes!
51 final attributes after extraction


In [431]:
df["price"] = df["price"].str.replace("GH₵ ", "").str.replace(",", "").astype(float)

In [432]:
def clean_self_contained(df):
    """
    Clean and fix the 'Self Contained' column based on sequential rules.
    """
    df = df.copy()

    print(f"NAs in self contained feature: {len(df[df['Self Contained'].isna()])}")

    sc_terms = [
        "Self Contained",
        "self-contained",
        "self-contain",
        "self-compound",
        "hall sc",
        "S/C",
        "self apartment",
        "acs",
        "aircondition",
        "self contain",
        "exacutive",
        "Self Compound",
        "Luxury",
        "apartment",
        "Newly",
        "Executive",
        "exective",
        "ultra",
        "modern",
        "furnished",
        "new",
        "suite",
        "ensuit",
        "en suite",
        "en-suite",
        "USD",
        "dollar",
        r"\$",
        "excellent",
        "Air condition",
        "Air-condition",
        "conditioning",
        "3bdrm",
        "4bdrm",
        "5bdrm",
        "6bdrm",
        "7bdrm",
        "8bdrm",
    ]
    nsc_terms = [
        "shared bathroom",
        "shared washroom",
        "shared toilet",
        "shared bath",
    ]

    sc_pattern = "|".join(sc_terms)
    nsc_pattern = "|".join(nsc_terms)

    sc_condition = (
        df["description"].str.contains(sc_pattern, case=False, na=False)
        | df["title"].str.contains(sc_pattern, case=False, na=False)
    ) & df["Self Contained"].isna()

    nsc_condition = (
        df["description"].str.contains(nsc_pattern, case=False, na=False)
        | df["title"].str.contains(nsc_pattern, case=False, na=False)
    ) & df["Self Contained"].isna()

    df.loc[sc_condition, "Self Contained"] = "Yes"
    print(
        f"Set 'Self Contained' to 'Yes' for {sc_condition.sum()} listings based on description/title and prior NaN status."
    )

    df.loc[nsc_condition, "Self Contained"] = "No"
    print(
        f"Set 'Self Contained' to 'No' for {nsc_condition.sum()} listings based on description/title and prior NaN status."
    )

    # 1. Rent is 2000 cedis +
    df.loc[
        (df["Self Contained"].isna()) & (df["price"] >= 2000),
        "Self Contained",
    ] = "Yes"

    # 2. Old condition AND Shared Apartment
    df.loc[
        (df["Self Contained"].isna())
        & (df["Condition"] == "Old")
        & (df["house_type"] == "Shared Apartment"),
        "Self Contained",
    ] = "No"

    # 3. Has amenities/furnishing
    df.loc[
        (df["Self Contained"].isna()) & (df["Wardrobe"] == 1)
        | (df["Wi-Fi"] == 1)
        | (df["TV"] == 1)
        | (df["Furnishing"] == "Furnished")
        | (df["Facilities"].notna())
        | (df["Dishwasher"] == 1)
        | (df["Dining Area"] == 1)
        | (df["Caution Fee"].notna()),
        "Self Contained",
    ] = "Yes"

    df.loc[
        (df["Self Contained"].isna()) & (df["price"] < 500),
        "Self Contained",
    ] = "No"

    df.loc[
        (df["Self Contained"].isna()) & (df["price"] >= 500),
        "Self Contained",
    ] = "Yes"

    print(f"NaNs Left: {len(df[df['Self Contained'].isna()])}")

    return df


df = clean_self_contained(df)

NAs in self contained feature: 8916
Set 'Self Contained' to 'Yes' for 8826 listings based on description/title and prior NaN status.
Set 'Self Contained' to 'No' for 18 listings based on description/title and prior NaN status.
NaNs Left: 0


In [433]:
def clean_toilets(df):
    df = df.copy()

    return df

In [434]:
df.isna().sum()


url                          0
description                  0
fetch_date                   0
title                        0
location                     0
house_type                   0
bathrooms                    0
bedrooms                     0
price                        0
Property Address            14
Condition                   45
Furnishing                   0
Estate Name               3837
Property Size               68
Agency Fee               10120
Toilets                   4275
Service Charge Fee       13745
Service Charge Covers    13915
Self Contained               0
Pets                     14460
Facilities               14743
Service Charge           14758
Caution Fee              12029
Minimum Rental Period     9443
Subtype                  12909
Listing by               14865
New Property             14867
Total Rooms              14863
Parking Space            14863
Smoking                  14864
Parties                  14864
Broker Fee               14864
Housing 

In [435]:
df.to_csv(OUTPUT)